# Hands-on 40 (Bab 15): Uji Beban serta Skenario Pengembangan Lanjut

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Empat pengujian realistis terhadap capstone: uji volume (laju tinggi), uji payload
rusak (field hilang, tipe salah, nilai ekstrem), uji anomali beruntun, dan uji drift
buatan. Diamati: stabilitas broker/subscriber, error handling, dan tanda drift.

## Tujuan

Pada latihan ini, pembaca akan:

- melakukan uji beban ringan,
- menguji pipeline dengan skenario data buruk,
- dan merancang pengembangan lanjut untuk capstone.

Ini sangat penting karena sistem yang tampak baik saat data normal masuk belum tentu tahan saat:

- data datang lebih cepat,
- payload rusak,
- anomali datang beruntun,
- atau distribusi data mulai bergeser.

## Skenario pengembangan lanjut

Setelah capstone dasar berjalan, arah pengembangan lanjut yang sangat masuk akal adalah:

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /downloads/iot-ml-book


## 1. Jalankan Uji Beban (volume + payload rusak + drift)

In [2]:
influx_hapus("capstone_device_monitoring")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/subscribers/capstone_subscriber.py", "stress_sub_nb.log")
    time.sleep(2)
    # uji volume: capstone_stress_publisher.py (skala buku: 1000 pesan, 0.05 detik)
    jalankan_buku("scripts/publishers/capstone_stress_publisher.py", 120)
    # uji payload rusak: capstone_bad_payload_publisher.py (3 payload buruk)
    jalankan_buku("scripts/publishers/capstone_bad_payload_publisher.py", 30)
    # uji drift: capstone_drift_publisher.py (skala buku: 500 pesan, 0.1 detik)
    jalankan_buku("scripts/publishers/capstone_drift_publisher.py", 120)
    time.sleep(3)
    henti(sub)
    out = log_teks("stress_sub_nb.log")
    lines = out.splitlines()
    print("\n".join(lines[:3]))
    print("...")
    n_processed = sum("anomaly=" in l for l in lines)
    n_errors = sum(l.startswith("Error:") for l in lines)
    print(f"\nPesan diproses: {n_processed}, error tertangani: {n_errors}")
    print("Baris terakhir subscriber:", lines[-1])
finally:
    broker.terminate()

Connected to broker.
Subscribed to capstone/iotnode01/telemetry
[2026-11-02T12:21:18] T=29.4 H=45.03 P=245.2 I=5.7 -> anomaly=0, threshold=0
...

Pesan diproses: 1021, error tertangani: 2
Baris terakhir subscriber: [2026-11-02T12:22:13] T=26.54 H=47.72 P=139.82 I=6.51 -> anomaly=0, threshold=0


## 2. Analisis Hasil Uji

In [3]:
df = influx_df("capstone_device_monitoring")
print(f"Total tersimpan: {len(df)} baris")
print(f"Anomali iforest: {df['iforest_anomaly'].sum()} ({df['iforest_anomaly'].mean()*100:.1f}%)")
print(f"Threshold alert: {df['threshold_alert'].sum()} ({df['threshold_alert'].mean()*100:.1f}%)")

# Tanda drift: bandingkan mean daya 50 baris pertama vs 50 baris terakhir
first = df["power_watt"].iloc[:50].mean()
last = df["power_watt"].iloc[-50:].mean()
print(f"\nMean power 50 awal: {first:.1f} W, 50 akhir: {last:.1f} W, geser {last-first:+.1f} W")

Total tersimpan: 54 baris
Anomali iforest: 10 (18.5%)
Threshold alert: 6 (11.1%)

Mean power 50 awal: 190.0 W, 50 akhir: 381.1 W, geser +191.2 W


## Jawaban dari data: Checklist hasil uji beban (Hands-on 40): jawaban dari hasil uji ini

In [4]:
import pandas as pd
df = influx_df("capstone_device_monitoring")
n = len(df)
print(f"1. Broker stabil: subscriber memproses {n_processed} pesan; yang tersimpan ({n} baris unik di InfluxDB (capstone_device_monitoring)); broker tidak berhenti selama uji.")
print("2. Subscriber tetap hidup: ia terus memproses pesan sampai publisher terakhir selesai (lihat keluaran uji di atas).")
print("3. Error handling bekerja: payload rusak dari publisher uji ditangani dan dihitung sebagai 'error tertangani' tanpa menghentikan subscriber (lihat keluaran uji).")
print(f"4. Storage masih menerima data: {n} baris unik tersimpan ke InfluxDB dari {n_processed} pesan diproses (pesan dengan timestamp detik yang sama menimpa satu sama lain).")
print("5. Dashboard: tidak dijalankan di notebook ini. Dashboard Grafana diuji terpisah (8 panel mengembalikan data); lihat README.")
ia, ta = int(df["iforest_anomaly"].sum()), int(df["threshold_alert"].sum())
print(f"6. Alert tidak terlalu berisik: Isolation Forest menandai {ia}/{n} ({ia/n*100:.1f}%), threshold {ta}/{n} ({ta/n*100:.1f}%).")
h = max(len(df) // 5, 1)
print(f"7. Tanda drift: rata-rata power {df['power_watt'].head(h).mean():.1f} W pada {h} pesan awal vs {df['power_watt'].tail(h).mean():.1f} W pada {h} pesan akhir (selisih {df['power_watt'].tail(h).mean()-df['power_watt'].head(h).mean():+.1f} W); pergeseran ini disengaja oleh publisher drift.")

1. Broker stabil: subscriber memproses 1021 pesan; yang tersimpan (54 baris unik di InfluxDB (capstone_device_monitoring)); broker tidak berhenti selama uji.
2. Subscriber tetap hidup: ia terus memproses pesan sampai publisher terakhir selesai (lihat keluaran uji di atas).
3. Error handling bekerja: payload rusak dari publisher uji ditangani dan dihitung sebagai 'error tertangani' tanpa menghentikan subscriber (lihat keluaran uji).
4. Storage masih menerima data: 54 baris unik tersimpan ke InfluxDB dari 1021 pesan diproses (pesan dengan timestamp detik yang sama menimpa satu sama lain).
5. Dashboard: tidak dijalankan di notebook ini. Dashboard Grafana diuji terpisah (8 panel mengembalikan data); lihat README.
6. Alert tidak terlalu berisik: Isolation Forest menandai 10/54 (18.5%), threshold 6/54 (11.1%).
7. Tanda drift: rata-rata power 197.1 W pada 10 pesan awal vs 1152.2 W pada 10 pesan akhir (selisih +955.1 W); pergeseran ini disengaja oleh publisher drift.


## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

**2. Jenis pengujian yang realistis**

Untuk buku ini, uji beban tidak perlu dibuat seperti benchmark enterprise. Cukup fokus pada empat hal:

- uji volume
- uji payload rusak
- uji anomali beruntun
- uji drift buatan

Mengapa empat ini?

Karena empat kondisi ini paling sering membuat pipeline lapangan mulai terlihat rapuh.

**3. Uji volume: percepat laju publisher**

Buat file scripts/publishers/capstone_stress_publisher.py:

```python
import json
import random
import time
from datetime import datetime
import paho.mqtt.client as mqtt
BROKER = "localhost"
PORT = 1883
TOPIC = "capstone/iotnode01/telemetry"
client = mqtt.Client(client_id="capstone-stress-publisher")
client.connect(BROKER, PORT, 60)
for i in range(1000):
    payload = {
        "timestamp":
datetime.now().isoformat(timespec="seconds"),
        "device_id": "iotnode01",
        "temperature_c": round(random.uniform(24.0, 34.0), 2),
        "humidity_pct": round(random.uniform(40.0, 75.0), 2),
        "power_watt": round(random.uniform(80.0, 260.0), 2),
        "current_amp": round(random.uniform(3.5, 9.5), 2)
    }
    if random.random() < 0.12:
        payload["power_watt"] += random.uniform(100, 220)
        payload["current_amp"] += random.uniform(2, 6)
    client.publish(TOPIC, json.dumps(payload), qos=0)
    if i % 100 == 0:
        print(f"Published {i} messages")
    time.sleep(0.05)
```

Apa yang harus diamati?

- apakah subscriber tetap stabil,
- apakah Grafana tetap menampilkan data,
- apakah InfluxDB tetap menerima data,
- apakah alert menjadi terlalu banyak.

**4. Uji payload rusak**

Buat file scripts/publishers/capstone_bad_payload_publisher.py:

```python
import json
import random
import time
from datetime import datetime
import paho.mqtt.client as mqtt
BROKER = "localhost"
TOPIC = "capstone/iotnode01/telemetry"
client = mqtt.Client(client_id="capstone-bad-publisher")
client.connect(BROKER, 1883, 60)
samples = [
    {"timestamp": datetime.now().isoformat(timespec="seconds"),
"device_id": "iotnode01"},
    {"timestamp": "bad_time", "device_id": "iotnode01",
"temperature_c": "abc", "humidity_pct": 50, "power_watt": 100,
"current_amp": 5},
    {"timestamp": datetime.now().isoformat(timespec="seconds"),
"device_id": "iotnode01", "temperature_c": -99, "humidity_pct":
999, "power_watt": 10000, "current_amp": -3},
]
for payload in samples:
    client.publish(TOPIC, json.dumps(payload), qos=1)
    print("Published bad payload:", payload)
    time.sleep(1)
```

Apa yang harus diamati?

- apakah subscriber menolak payload rusak,
- apakah error tercatat,
- apakah sistem tetap jalan setelah menerima data rusak.

Ini sangat sesuai dengan topik bab operasional sebelumnya tentang data palsu, data hilang, data rusak dan perlunya validasi input.

**5. Uji drift buatan**

Buat file scripts/publishers/capstone_drift_publisher.py:

```python
import json
import random
import time
from datetime import datetime
import paho.mqtt.client as mqtt
BROKER = "localhost"
TOPIC = "capstone/iotnode01/telemetry"
client = mqtt.Client(client_id="capstone-drift-publisher")
client.connect(BROKER, 1883, 60)
base_temp = 26.0
base_power = 130.0
for i in range(500):
    # drift perlahan
    temp = base_temp + (i * 0.01) + random.uniform(-0.5, 0.5)
    power = base_power + (i * 0.08) + random.uniform(-10, 10)
    payload = {
        "timestamp":
datetime.now().isoformat(timespec="seconds"),
        "device_id": "iotnode01",
        "temperature_c": round(temp, 2),
        "humidity_pct": round(random.uniform(45.0, 65.0), 2),
        "power_watt": round(power, 2),
        "current_amp": round(random.uniform(4.0, 7.5), 2)
    }
     client.publish(TOPIC, json.dumps(payload), qos=1)
     print("Published drift payload:", payload)
     time.sleep(0.1)
```

Apa yang harus diamati?

● apakah distribusi data bergeser di dashboard, ● apakah model mulai lebih sering memberi anomaly, ● apakah sistem butuh retraining. Ini berkaitan langsung dengan kebutuhan monitoring drift dan penurunan performa model

**6. Checklist hasil uji beban**

Setelah pengujian, pembaca sebaiknya menjawab:

- apakah broker tetap stabil?
- apakah subscriber tetap hidup?
- apakah error handling bekerja?
- apakah storage masih menerima data?
- apakah dashboard tetap bisa dipakai?
- apakah alert masih berguna atau jadi terlalu berisik?
- apakah ada tanda drift yang perlu ditangani?

**7.1. Multi-device**

Dari satu node menjadi banyak node:

- iotnode01
- iotnode02
- lab01
- room01

**7.2. Multi-sensor fusion**

Tambah:

- CO₂
- vibration
- voltage
- GPS
- soil moisture

**7.3. Edge deployment**

Pindahkan subscriber + inferensi ke:

- Raspberry Pi
- mini PC edge
- gateway lokal

Ini sejalan dengan pembahasan edge AI dan inferensi lokal pada bab sebelumnya.

**7.4. Retraining berkala**

Tambahkan:

- batch retraining mingguan
- versioning model
- evaluasi drift berkala

**7.5. Security hardening**

Tambahkan:

- autentikasi MQTT
- pemisahan topik
- TLS bila perlu

**7.6. Docker dan reproducibility**

Setelah sistem dipahami, bungkus:

- Mosquitto
- InfluxDB
- Grafana
- subscriber

ke dalam docker-compose.yml

**7.7. Explainability ringan**

Tambahkan:

- feature importance
- narasi penyebab alert
- log “kenapa alert dipicu”

**8. Bentuk hasil akhir capstone yang ideal**

Jika Hands-on 38–40 berjalan baik, maka hasil akhir proyek seharusnya memiliki:

- simulator atau sensor nyata yang mengirim data,
- broker MQTT yang stabil,
- subscriber Python dengan preprocessing otomatis,
- model anomaly detection yang bekerja,
- histori data di storage,
- dashboard Grafana yang hidup,
- alert sederhana,
- hasil uji beban dasar,
- dan daftar arah pengembangan lanjut.

Itulah bentuk Sistem IoT-ML Open Source Lengkap yang realistis, bukan hanya demo sekali jalan.

## Penutup



## Penutup - Checklist Hasil Uji Beban

Angka hasil uji ada pada keluaran sel di atas dan di bagian "Jawaban dari data" di bawah (dihitung dari proses nyata, bukan ditulis tangan).

Catatan jujur: skrip buku menulis ke InfluxDB dengan timestamp presisi detik dan tag yang sama.
Pesan yang tiba dalam detik yang sama saling menimpa, sehingga jumlah baris di InfluxDB lebih kecil
daripada jumlah pesan yang diproses subscriber pada uji volume cepat (1000 pesan, 0,05 detik).

Pengembangan lanjut sesuai buku: multi-device, multi-model, retraining terjadwal,
notifikasi eksternal, dan hardening keamanan MQTT.
